In [ ]:
import pandas as pd
#加载数据
df = pd.read_csv('../数据/1.标签构造后的数据.csv')

#显示数据的基本信息
print("数据集形状:", df.shape)
print("\n前5行数据预览:")
print(df.head())

print("\n数据列名:")
print(df.columns.tolist())

In [ ]:
# 缺失值预处理与基于直方图梯度提升的填充
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier

target = 'ms_cds'

#删除缺失率 > 60% 的列
print("正在检查并删除缺失率超过60%的列...")
missing_ratio = df.drop(columns=[target]).isnull().mean()
cols_to_drop_60 = missing_ratio[missing_ratio > 0.6].index.tolist()
if cols_to_drop_60:
    print(f"将删除以下 {len(cols_to_drop_60)} 列（缺失率 > 60%）：{cols_to_drop_60}")
    df = df.drop(columns=cols_to_drop_60)
else:
    print("未发现缺失率超过60%的列。")

#删除缺失比例过高的行
row_missing_threshold = 0.4
non_target_cols = [c for c in df.columns if c != target]
row_missing_ratio = df[non_target_cols].isnull().mean(axis=1)
rows_to_drop = row_missing_ratio > row_missing_threshold
if rows_to_drop.sum() > 0:
    print(f"删除缺失率超过 {row_missing_threshold:.0%} 的行：{rows_to_drop.sum()} 行")
    df = df[~rows_to_drop].copy()
else:
    print("没有因行缺失率过高而删除的样本。")


#分离特征与标签
X = df.drop(columns=[target]).copy()
y = df[target].copy()

#区分连续变量与分类变量
categorical_cols = [
    col for col in X.columns
    if X[col].nunique() < 20
]
continuous_cols = [col for col in X.columns if col not in categorical_cols]
print(f"连续变量 {len(continuous_cols)} 个，分类变量 {len(categorical_cols)} 个")

#确保分类变量为整数
for col in categorical_cols:

    if X[col].dtype == 'float64':

        mask_notna = X[col].notna()
        X.loc[mask_notna, col] = X.loc[mask_notna, col].astype(int)

#迭代梯度提升填充
def gb_native_imputer(X, continuous_cols, categorical_cols, max_iter=5, random_state=42):
    X_filled = X.copy()
    all_cols = continuous_cols + categorical_cols

    col_missing = X_filled[all_cols].isnull().sum()
    sorted_cols = col_missing[col_missing > 0].sort_values().index.tolist()

    if not sorted_cols:
        print("没有需要填充的缺失值")
        return X_filled

    for iteration in range(max_iter):
        print(f"--- 迭代 {iteration+1}/{max_iter} ---")
        for target_col in sorted_cols:
            missing_mask = X_filled[target_col].isna()
            if missing_mask.sum() == 0:
                continue

            feature_cols = [c for c in all_cols if c != target_col]
            X_train = X_filled.loc[~missing_mask, feature_cols]
            y_train = X_filled.loc[~missing_mask, target_col]
            X_pred = X_filled.loc[missing_mask, feature_cols]

            valid_y = y_train.notna()
            if not valid_y.all():
                y_train = y_train[valid_y]
                X_train = X_train[valid_y]

            if target_col in continuous_cols:
                model = HistGradientBoostingRegressor(
                    max_iter=100, max_depth=5, random_state=random_state
                )
                model.fit(X_train, y_train)
                preds = model.predict(X_pred)
                lower, upper = y_train.min(), y_train.max()
                preds = np.clip(preds, lower, upper)
            else:
                model = HistGradientBoostingClassifier(
                    max_iter=100, max_depth=5, random_state=random_state
                )
                y_train_int = y_train.astype(int)
                model.fit(X_train, y_train_int)
                preds = model.predict(X_pred)
                lower, upper = y_train_int.min(), y_train_int.max()
                preds = np.clip(preds, lower, upper).round().astype(int)

            X_filled.loc[missing_mask, target_col] = preds

    return X_filled

print("开始基于梯度提升原生缺失处理的迭代填充")
X_filled = gb_native_imputer(
    X, continuous_cols, categorical_cols,
    max_iter=5, random_state=42
)

df_cleaned = X_filled.copy()
df_cleaned[target] = y.values
df_cleaned.describe()[['ggt', 'ast', 'age', 'hr', 'alt', 'mono', 'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc']].to_excel('describe.xlsx')
print(f"填充完成。最终数据维度：{df_cleaned.shape}")
print(f"剩余缺失值数量：{df_cleaned.isnull().sum().sum()}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
sns.set_palette("husl")

target = 'ms_cds'
tezheng = ['ggt', 'drinking', 'ast', 'age', 'hr', 'alt', 'mono', 'gender', 'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc',target]
df_sub = df_cleaned[tezheng].copy()
df_sub.rename(columns={'ms_cds': 'target'}, inplace=True)
# 计算相关系数矩阵
corr = df_sub.corr(method='pearson')
# 设置图形大小
plt.figure(figsize=(10, 8))

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
# 绘制热力图
ax = sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
ax.set_title(' Heat map of some features and targets', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('热力图.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = df_cleaned

existing_cont = [col for col in continuous_cols if col in df.columns]
print(f"实际存在的连续特征数量: {len(existing_cont)}")

df_melted = df[existing_cont].melt(var_name='feature', value_name='value')
plt.figure(figsize=(20, 10))
sns.boxplot(data=df_melted, x='feature', y='value')
plt.xticks(rotation=90)
plt.title('Boxplots of All Continuous Features (Combined)')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['legend.title_fontsize'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['axes.linewidth'] = 1.0

sns.set_style("whitegrid")
sns.set_palette("Set1")

#  数据准备 
features = ['ggt', 'mono', 'hr', 'age', 'dbil', 'rbc']
df_plot = df_cleaned[features + [target]].dropna().copy()
target_counts = df_plot[target].value_counts().sort_index()

class_labels = df_plot[target].unique()
palette_colors = sns.color_palette("Set1")

#  绘图 
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]

    sns.kdeplot(
        data=df_plot, x=col, hue=target,
        fill=True,
        alpha=0.2,
        linewidth=1.5,
        common_norm=False,
        ax=ax,
        legend=False
    )

    ax.set_title(f'Kernel Density of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.grid(alpha=0.4)

    legend_elements = []
    for idx, label in enumerate(class_labels):
        color = palette_colors[idx % len(palette_colors)]
        n_count = target_counts.get(label, 0)
        legend_line = Line2D([0], [0], color=color, lw=1.5, label=f'{label} (N={n_count})')
        legend_elements.append(legend_line)

    ax.legend(handles=legend_elements, title=target, loc='upper right',
              frameon=True, fancybox=False, edgecolor='black', facecolor='white')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

fig.savefig('核密度图.png', dpi=600)
plt.show()

## 鲁棒标准化

In [ ]:
from sklearn.preprocessing import RobustScaler
df = df_cleaned

scaler = RobustScaler()

df_scaled = df.copy()
df_scaled[continuous_cols] = scaler.fit_transform(df[continuous_cols])

df_scaled = df_scaled.drop_duplicates()
df_scaled[['ggt', 'drinking', 'ast', 'age', 'hr', 'alt', 'mono', 'gender', 'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc']].to_excel('鲁棒标准化后的数据.xlsx', index=False)



import pandas as pd
from sklearn.preprocessing import RobustScaler

df = df_cleaned
#后续选择的特征的中位数与四分位数表，用于后于shap分析还原原始值
continuous_features = ['ggt', 'ast', 'age', 'hr', 'alt', 'mono', 'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc']

scaler = RobustScaler()
scaler.fit(df[continuous_features])

# 提取中位数和四分位距
medians = scaler.center_
iqrs = scaler.scale_

# 创建 DataFrame 展示结果
stats_df = pd.DataFrame({
    'feature': continuous_features,
    'median': medians,
    'iqr': iqrs
})
stats_df.to_excel('feature_median_iqr.xlsx', index=False)

## 划分训练集和测试集

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_scaled, test_size=0.2, random_state=42)
train_df.to_csv('../数据/训练集.csv', index=False)
test_df.to_csv('../数据/测试集.csv', index=False)


In [ ]:
df_scaled[target].value_counts()